## 1. **Import Libraries**

In [ ]:
from pathlib import Path
import json
import random
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)
from tensorflow.keras.applications import EfficientNetV2S

# Mixed-precision (float16) for faster GPU training
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices("GPU")) > 0)
print("Compute dtype :", policy.compute_dtype)
print("Variable dtype:", policy.variable_dtype)


## 2. **Data Loading and Preprocessing**

In [ ]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
PROCESSED_DATA_DIR = Path('/content/drive/MyDrive/Brain-Tumor-Mri-Deep-Learning/data/processed')
TRAIN_DIR = PROCESSED_DATA_DIR / "train"
VAL_DIR   = PROCESSED_DATA_DIR / "val"
TEST_DIR  = PROCESSED_DATA_DIR / "test"

PROJECT_ROOT = Path(".")
MODEL_DIR  = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
LOG_DIR    = PROJECT_ROOT / "logs"

for d in [MODEL_DIR, REPORT_DIR, FIGURE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
IMG_HEIGHT = IMG_SIZE[0]
IMG_WIDTH  = IMG_SIZE[1]
CHANNELS   = 3

BATCH_SIZE    = 32
EPOCHS_PHASE1 = 10   # head-only warm-up
EPOCHS_PHASE2 = 60   # full fine-tune
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Train path : {TRAIN_DIR}")
print(f"Val path   : {VAL_DIR}")
print(f"Test path  : {TEST_DIR}")


In [ ]:
required_dirs = {"train": TRAIN_DIR, "validation": VAL_DIR, "test": TEST_DIR}
for split_name, dir_path in required_dirs.items():
    if not dir_path.exists():
        raise FileNotFoundError(f"Khong tim thay folder {split_name}: {dir_path}")
print("Tat ca folder train/val/test da ton tai.")


In [ ]:
def count_images_by_class(directory):
    image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
    records = []
    for class_dir in sorted(directory.iterdir()):
        if class_dir.is_dir():
            image_count = sum(len(list(class_dir.glob(ext))) for ext in image_extensions)
            records.append({"class_name": class_dir.name, "image_count": image_count})
    return pd.DataFrame(records)

train_count_df = count_images_by_class(TRAIN_DIR)
val_count_df   = count_images_by_class(VAL_DIR)
test_count_df  = count_images_by_class(TEST_DIR)

print("TRAIN DATASET"); display(train_count_df)
print("VALIDATION DATASET"); display(val_count_df)
print("TEST DATASET"); display(test_count_df)
print("Total train images      :", train_count_df["image_count"].sum())
print("Total validation images :", val_count_df["image_count"].sum())
print("Total test images       :", test_count_df["image_count"].sum())


In [ ]:
def plot_class_distribution(df, title, save_path):
    plt.figure(figsize=(14, 5))
    plt.bar(df["class_name"], df["image_count"])
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

plot_class_distribution(train_count_df, "Train Dataset Distribution",
                        FIGURE_DIR / "train_distribution.png")
plot_class_distribution(val_count_df,   "Validation Dataset Distribution",
                        FIGURE_DIR / "val_distribution.png")
plot_class_distribution(test_count_df,  "Test Dataset Distribution",
                        FIGURE_DIR / "test_distribution.png")


In [ ]:
# EfficientNetV2S expects pixel values in [0, 255].
# Its include_preprocessing=True handles normalisation internally.
# We use preprocessing_function from tf.keras.applications.efficientnet_v2
# for the ImageDataGenerator pipeline.

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.12,
    height_shift_range=0.12,
    zoom_range=0.12,
    horizontal_flip=True,
    vertical_flip=False,           # MRI brain scans are typically upright
    brightness_range=[0.8, 1.2],   # simulate MRI contrast variation
    fill_mode="nearest",
    preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input
)

train_data = train_datagen.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_data = val_test_datagen.flow_from_directory(
    directory=VAL_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_datagen.flow_from_directory(
    directory=TEST_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_data.num_classes
CLASS_NAMES  = list(train_data.class_indices.keys())
print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)


In [ ]:
images, labels = next(train_data)

plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    # De-normalise for display (EfficientNetV2 preprocess scales to [-1, 1])
    img = (images[i] + 1.0) / 2.0
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    class_index = np.argmax(labels[i])
    plt.title(CLASS_NAMES[class_index], fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "preprocessed_sample_images.png")
plt.show()

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


### Compute class weights (handle class imbalance)

Brain-tumour datasets are often skewed (rare subtypes have fewer MRI scans).
Weighted loss ensures the model pays equal attention to every tumour type,
which is crucial for clinical utility and for achieving high macro-F1.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = train_data.classes   # integer class indices for all training images

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
class_weights = {i: float(w) for i, w in enumerate(class_weights_array)}

print("Class weights (first 5):", dict(list(class_weights.items())[:5]))
print("Max weight :", max(class_weights.values()))
print("Min weight :", min(class_weights.values()))


## **Build Model – EfficientNetV2S + Transfer Learning**

### Architecture rationale

| Component | Choice | Why |
|-----------|--------|-----|
| Backbone | EfficientNetV2S (ImageNet) | SOTA accuracy-per-parameter; fused-MBConv blocks train fast |
| Head | GAP -> BN -> Dense(512, swish) -> Dropout(0.4) -> Dense(256, swish) -> Dropout(0.3) -> Dense(30, softmax) | Small head avoids overfitting on ~11k medical images |
| Loss | CategoricalCrossentropy + label_smoothing=0.1 | Prevents overconfident predictions; reduces overfitting |
| Optimiser phase 1 | Adam lr=1e-3 | Fast convergence for head-only training |
| Optimiser phase 2 | Adam + CosineDecayRestarts lr=2e-4 to 1e-6 | Smooth annealing avoids sharp overfitting during fine-tune |
| Regularisation | Dropout 0.4/0.3, L2(1e-4) on Dense, class weights | Multi-level defence against overfitting |

### Two-phase training strategy
1. **Phase 1 – Warm-up (10 epochs)**: Backbone frozen. Only the custom head is trained.
2. **Phase 2 – Fine-tune (up to 60 epochs)**: Top 80 layers of backbone unfrozen with very low LR.


In [ ]:
from tensorflow.keras import regularizers

def build_efficientnet_model(input_shape, num_classes):
    # ── Backbone ──────────────────────────────────────────────────────────
    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        include_preprocessing=True    # built-in normalisation layer
    )
    base_model.trainable = False      # freeze during phase 1

    # ── Custom head ───────────────────────────────────────────────────────
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = base_model(inputs, training=False)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_head")(x)

    x = layers.Dense(
        512, activation="swish",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_512"
    )(x)
    x = layers.Dropout(0.4, name="drop_1")(x)

    x = layers.Dense(
        256, activation="swish",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_256"
    )(x)
    x = layers.Dropout(0.3, name="drop_2")(x)

    # dtype=float32 for numerical stability with mixed-precision training
    outputs = layers.Dense(
        num_classes, activation="softmax",
        dtype="float32",
        name="predictions"
    )(x)

    model = tf.keras.Model(inputs, outputs, name="BrainTumor_EfficientNetV2S")
    return model, base_model


input_shape = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)
model, base_model = build_efficientnet_model(input_shape, NUM_CLASSES)
model.summary()
print(f"\nTotal params     : {model.count_params():,}")


## **Phase 1 – Head Warm-up (backbone frozen)**

In [ ]:
# Shared loss with label smoothing
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss_fn,
    metrics=["accuracy"]
)

best_model_path  = MODEL_DIR / "efficientnet_best.keras"
final_model_path = MODEL_DIR / "efficientnet_final.keras"
log_phase1_path  = LOG_DIR / "phase1_log.csv"

callbacks_phase1 = [
    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger(filename=log_phase1_path, append=False)
]

print("=== PHASE 1: Training head only ===")
history_phase1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
    class_weight=class_weights
)


## **Phase 2 – Fine-tuning (unfreeze top 80 backbone layers)**

In [ ]:
# Unfreeze top 80 layers of EfficientNetV2S
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 80

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable backbone layers: {trainable_count} / {len(base_model.layers)}")

# Cosine decay with warm restarts – smooth LR, prevents sharp overfitting
total_steps   = len(train_data) * EPOCHS_PHASE2
lr_schedule   = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=2e-4,
    first_decay_steps=total_steps // 3,
    t_mul=1.0,
    m_mul=0.9,
    alpha=1e-6
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=loss_fn,
    metrics=["accuracy"]
)

log_phase2_path = LOG_DIR / "phase2_log.csv"

callbacks_phase2 = [
    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger(filename=log_phase2_path, append=False)
]

print("=== PHASE 2: Fine-tuning top 80 backbone layers ===")
history_phase2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2,
    class_weight=class_weights
)

model.save(final_model_path)
print(f"Final model saved: {final_model_path}")


## **Plot Training History**

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

merged    = merge_histories(history_phase1, history_phase2)
phase1_end = len(history_phase1.history["accuracy"])

def plot_training_history(merged, phase1_end):
    acc      = merged["accuracy"]
    val_acc  = merged["val_accuracy"]
    loss     = merged["loss"]
    val_loss = merged["val_loss"]
    epochs_range = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, acc,     label="Train Accuracy")
    axes[0].plot(epochs_range, val_acc, label="Val Accuracy")
    axes[0].axvline(phase1_end, color="gray", linestyle="--", label="Phase 1 -> 2")
    axes[0].set_title("EfficientNetV2S – Accuracy Curve")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs_range, loss,     label="Train Loss")
    axes[1].plot(epochs_range, val_loss, label="Val Loss")
    axes[1].axvline(phase1_end, color="gray", linestyle="--", label="Phase 1 -> 2")
    axes[1].set_title("EfficientNetV2S – Loss Curve")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "efficientnet_training_curves.png", dpi=150)
    plt.show()

plot_training_history(merged, phase1_end)


## **Evaluation & Metrics**

In [ ]:
# Load best checkpoint for evaluation
best_model = tf.keras.models.load_model(best_model_path)

test_loss, test_accuracy = best_model.evaluate(test_data, verbose=1)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")


In [ ]:
test_data.reset()
y_pred_prob = best_model.predict(test_data, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_data.classes

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)


In [ ]:
accuracy           = accuracy_score(y_true, y_pred)
precision_macro    = precision_score(y_true, y_pred, average="macro",    zero_division=0)
recall_macro       = recall_score(y_true,    y_pred, average="macro",    zero_division=0)
f1_macro           = f1_score(y_true,        y_pred, average="macro",    zero_division=0)
precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted    = recall_score(y_true,    y_pred, average="weighted", zero_division=0)
f1_weighted        = f1_score(y_true,        y_pred, average="weighted", zero_division=0)

metrics_summary = {
    "model_name":         "EfficientNetV2S (fine-tuned)",
    "image_size":         str(IMG_SIZE),
    "batch_size":         BATCH_SIZE,
    "epochs_phase1":      len(history_phase1.history["loss"]),
    "epochs_phase2":      len(history_phase2.history["loss"]),
    "test_loss":          float(test_loss),
    "test_accuracy":      float(accuracy),
    "precision_macro":    float(precision_macro),
    "recall_macro":       float(recall_macro),
    "f1_macro":           float(f1_macro),
    "precision_weighted": float(precision_weighted),
    "recall_weighted":    float(recall_weighted),
    "f1_weighted":        float(f1_weighted)
}

metrics_df = pd.DataFrame([metrics_summary])
display(metrics_df)
metrics_df.to_csv(REPORT_DIR / "efficientnet_metrics_summary.csv", index=False)


In [ ]:
print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# Confusion matrix – counts and normalised side by side
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Confusion Matrix (normalised)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "efficientnet_confusion_matrix.png", dpi=150)
plt.show()
